
# Generic Health Insurance EDA & Pattern-Based Rule Mining

Reusable for claims fraud, leakage, denial, high-cost claims, provider abuse, churn, lapse, grievance escalation, underwriting risk, utilization anomalies, and operational delays.

The notebook performs file loading, data profiling, missing-value handling, feature typing, health-insurance feature engineering, univariate/bivariate EDA, numeric binning, exhaustive combination mining, FP-Growth association rules, decision-tree rules, holdout validation, risk bands, and Excel/CSV export.

> Discovered rules are associations, not proof of causation. Review for leakage, stability, fairness, regulatory acceptability, and operational usefulness before deployment.


## 1. Install and import libraries

In [ ]:

%pip install -q pandas numpy openpyxl scikit-learn matplotlib mlxtend scipy


In [ ]:

import os, re, math, warnings
from itertools import combinations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier, export_text, _tree
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score
from mlxtend.frequent_patterns import fpgrowth, association_rules
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', 300)


## 2. Configuration

In [ ]:

FILE_PATH = 'your_uploaded_file.xlsx'
SHEET_NAME = 0
TARGET_COLUMN = 'fraud_reported'
POSITIVE_CLASS = 'Y'

ID_COLUMNS = []
DATE_COLUMNS = []
FREE_TEXT_COLUMNS = []
FORCE_CATEGORICAL = []
FORCE_NUMERIC = []
EXCLUDE_COLUMNS = []

HEALTH_FIELD_MAP = {
    'claim_amount': None,
    'premium_amount': None,
    'sum_insured': None,
    'policy_start_date': None,
    'claim_date': None,
    'admission_date': None,
    'discharge_date': None,
    'member_age': None,
    'hospital_id': None,
    'provider_city': None,
    'diagnosis_code': None,
    'procedure_code': None,
    'room_rent': None,
    'length_of_stay': None,
    'network_flag': None,
    'cashless_flag': None,
    'claim_type': None,
}

MIN_SUPPORT_COUNT = 20
MIN_POSITIVE_COUNT = 5
MIN_CONFIDENCE = 0.35
MIN_LIFT = 1.30
MAX_RULE_LENGTH = 3
TOP_N_CATEGORIES = 20
MAX_NUMERIC_BINS = 5

ASSOCIATION_MIN_SUPPORT = 0.02
ASSOCIATION_MIN_CONFIDENCE = 0.35
ASSOCIATION_MIN_LIFT = 1.30
ASSOCIATION_MAX_LEN = 4

TREE_MAX_DEPTH = 5
TREE_MIN_SAMPLES_LEAF = 20
TEST_SIZE = 0.25
RANDOM_STATE = 42
OUTPUT_FOLDER = 'pattern_mining_outputs'
OUTPUT_EXCEL = 'health_insurance_pattern_rules.xlsx'
os.makedirs(OUTPUT_FOLDER, exist_ok=True)


## 3. Load and standardize data

In [ ]:

def load_data(path, sheet_name=0):
    ext = os.path.splitext(path)[1].lower()
    if ext in ['.xlsx', '.xls']:
        return pd.read_excel(path, sheet_name=sheet_name)
    if ext == '.csv':
        return pd.read_csv(path)
    if ext in ['.parquet', '.pq']:
        return pd.read_parquet(path)
    raise ValueError(f'Unsupported file type: {ext}')

def clean_name(x):
    return re.sub(r'[^a-z0-9]+', '_', str(x).strip().lower()).strip('_')

df_raw = load_data(FILE_PATH, SHEET_NAME)
df = df_raw.copy()
df.columns = [clean_name(c) for c in df.columns]
TARGET_COLUMN = clean_name(TARGET_COLUMN)
ID_COLUMNS = [clean_name(c) for c in ID_COLUMNS]
DATE_COLUMNS = [clean_name(c) for c in DATE_COLUMNS]
FREE_TEXT_COLUMNS = [clean_name(c) for c in FREE_TEXT_COLUMNS]
FORCE_CATEGORICAL = [clean_name(c) for c in FORCE_CATEGORICAL]
FORCE_NUMERIC = [clean_name(c) for c in FORCE_NUMERIC]
EXCLUDE_COLUMNS = [clean_name(c) for c in EXCLUDE_COLUMNS]
for k,v in list(HEALTH_FIELD_MAP.items()):
    if v: HEALTH_FIELD_MAP[k] = clean_name(v)

df = df.replace(['?', '', 'NA', 'N/A', 'NULL', 'null', 'None', 'none', '-', '--'], np.nan)
for c in df.select_dtypes(include='object').columns:
    df[c] = df[c].astype('string').str.strip().replace({'': pd.NA})
for c in DATE_COLUMNS:
    if c in df.columns: df[c] = pd.to_datetime(df[c], errors='coerce')
print('Shape:', df.shape)
display(df.head())


## 4. Data profile and target preparation

In [ ]:

def data_profile(data):
    rows=[]
    for c in data.columns:
        s=data[c]
        rows.append({'column':c,'dtype':str(s.dtype),'rows':len(s),'non_null':int(s.notna().sum()),
                     'missing':int(s.isna().sum()),'missing_pct':round(s.isna().mean()*100,2),
                     'unique':int(s.nunique(dropna=True)),'unique_pct':round(s.nunique(dropna=True)/max(len(s),1)*100,2),
                     'sample_values':', '.join(map(str,s.dropna().astype(str).unique()[:5]))})
    return pd.DataFrame(rows).sort_values(['missing_pct','unique'],ascending=[False,False])

profile=data_profile(df)
display(profile)
if TARGET_COLUMN not in df.columns: raise KeyError(f"Target '{TARGET_COLUMN}' not found")
df=df[df[TARGET_COLUMN].notna()].copy()
if pd.api.types.is_numeric_dtype(df[TARGET_COLUMN]):
    df['target_flag']=(df[TARGET_COLUMN]==POSITIVE_CLASS).astype(int)
else:
    df['target_flag']=(df[TARGET_COLUMN].astype(str).str.strip().str.lower()==str(POSITIVE_CLASS).strip().lower()).astype(int)
base_rate=df['target_flag'].mean()
print('Positive count:', int(df.target_flag.sum()), 'Base rate:', round(base_rate,4))


## 5. Feature typing and health-insurance features

In [ ]:

def infer_types(data):
    excluded=set([TARGET_COLUMN,'target_flag']+ID_COLUMNS+FREE_TEXT_COLUMNS+EXCLUDE_COLUMNS)
    num=[]; cat=[]; dates=[]
    for c in [x for x in data.columns if x not in excluded]:
        if c in DATE_COLUMNS or pd.api.types.is_datetime64_any_dtype(data[c]): dates.append(c)
        elif c in FORCE_CATEGORICAL: cat.append(c)
        elif c in FORCE_NUMERIC or pd.api.types.is_numeric_dtype(data[c]): num.append(c)
        else: cat.append(c)
    return num,cat,dates

def safe_ratio(a,b): return a/b.replace(0,np.nan)
def add_health_features(data,f):
    out=data.copy()
    def ok(k): return f.get(k) and f[k] in out.columns
    if ok('claim_amount') and ok('premium_amount'):
        out['claim_to_premium_ratio']=safe_ratio(pd.to_numeric(out[f['claim_amount']],errors='coerce'),pd.to_numeric(out[f['premium_amount']],errors='coerce'))
    if ok('claim_amount') and ok('sum_insured'):
        out['claim_to_sum_insured_ratio']=safe_ratio(pd.to_numeric(out[f['claim_amount']],errors='coerce'),pd.to_numeric(out[f['sum_insured']],errors='coerce'))
    if ok('policy_start_date') and ok('claim_date'):
        out['policy_tenure_days_at_claim']=(pd.to_datetime(out[f['claim_date']],errors='coerce')-pd.to_datetime(out[f['policy_start_date']],errors='coerce')).dt.days
    if ok('admission_date') and ok('discharge_date'):
        out['calculated_length_of_stay']=(pd.to_datetime(out[f['discharge_date']],errors='coerce')-pd.to_datetime(out[f['admission_date']],errors='coerce')).dt.days
    if ok('room_rent') and ok('claim_amount'):
        out['room_rent_share']=safe_ratio(pd.to_numeric(out[f['room_rent']],errors='coerce'),pd.to_numeric(out[f['claim_amount']],errors='coerce'))
    if ok('member_age'):
        age=pd.to_numeric(out[f['member_age']],errors='coerce')
        out['member_age_band']=pd.cut(age,[-np.inf,18,30,45,60,75,np.inf],labels=['<=18','19-30','31-45','46-60','61-75','76+']).astype('string')
    return out

df=add_health_features(df,HEALTH_FIELD_MAP)
numeric_cols,categorical_cols,date_cols=infer_types(df)
for c in numeric_cols+categorical_cols:
    if df[c].isna().any(): df[f'{c}__missing']=df[c].isna().astype(int)
numeric_cols,categorical_cols,date_cols=infer_types(df)
print('Numeric:',len(numeric_cols),'Categorical:',len(categorical_cols),'Dates:',len(date_cols))


## 6. Numeric and categorical EDA

In [ ]:

def numeric_eda(data,cols):
    rows=[]
    for c in cols:
        s=pd.to_numeric(data[c],errors='coerce')
        if s.notna().sum()==0: continue
        for cls in [0,1]:
            x=s[data.target_flag==cls]
            rows.append({'feature':c,'target':cls,'count':int(x.notna().sum()),'missing':int(x.isna().sum()),
                         'mean':x.mean(),'median':x.median(),'std':x.std(),'min':x.min(),'p25':x.quantile(.25),'p75':x.quantile(.75),'max':x.max()})
    return pd.DataFrame(rows)

def wilson(successes,total):
    if total==0:return np.nan,np.nan
    z=1.959963984540054;p=successes/total;d=1+z*z/total
    center=(p+z*z/(2*total))/d;margin=z*math.sqrt((p*(1-p)+z*z/(4*total))/total)/d
    return center-margin,center+margin

def categorical_lift(data,feature):
    t=data[[feature,'target_flag']].copy();t[feature]=t[feature].astype('string').fillna('UNKNOWN')
    o=t.groupby(feature,dropna=False).target_flag.agg(['count','sum']).reset_index()
    o.columns=[feature,'records','positives'];o['positive_rate']=o.positives/o.records;o['lift']=o.positive_rate/max(data.target_flag.mean(),1e-9)
    ci=o.apply(lambda r:wilson(r.positives,r.records),axis=1);o['ci_low']=[x[0] for x in ci];o['ci_high']=[x[1] for x in ci]
    o['feature']=feature;o['value']=o[feature].astype(str)
    return o[['feature','value','records','positives','positive_rate','lift','ci_low','ci_high']]

numeric_summary=numeric_eda(df,numeric_cols)
uni=[categorical_lift(df,c) for c in categorical_cols if df[c].nunique(dropna=True)<=500]
univariate_rules=pd.concat(uni,ignore_index=True).sort_values(['lift','records'],ascending=[False,False]) if uni else pd.DataFrame()
display(numeric_summary.head(30));display(univariate_rules.head(100))


## 7. Prepare rule-mining data

In [ ]:

def discretize(data,cols,max_bins=5):
    out=pd.DataFrame(index=data.index);meta=[]
    for c in cols:
        s=pd.to_numeric(data[c],errors='coerce')
        if s.nunique(dropna=True)<2:continue
        try:
            b,e=pd.qcut(s,q=min(max_bins,s.nunique()),duplicates='drop',retbins=True)
            out[f'{c}__bin']=b.astype('string').fillna('UNKNOWN');meta.append({'feature':c,'edges':e.tolist()})
        except Exception:
            out[f'{c}__bin']=pd.cut(s,bins=max_bins,duplicates='drop').astype('string').fillna('UNKNOWN')
    return out,pd.DataFrame(meta)

numeric_binned,bin_metadata=discretize(df,numeric_cols,MAX_NUMERIC_BINS)
rule_frame=pd.DataFrame(index=df.index)
for c in categorical_cols:
    v=df[c].astype('string').fillna('UNKNOWN')
    if v.nunique()>TOP_N_CATEGORIES:
        top=v.value_counts().head(TOP_N_CATEGORIES).index;v=v.where(v.isin(top),'OTHER')
    rule_frame[c]=v
rule_frame=pd.concat([rule_frame,numeric_binned],axis=1)
rule_frame['target_flag']=df.target_flag.values
print(rule_frame.shape)


## 8. Exhaustive combination mining

In [ ]:

def mine_combinations(data,max_len=3,min_support=20,min_pos=5,min_conf=.35,min_lift=1.3):
    features=[c for c in data.columns if c!='target_flag'];base=data.target_flag.mean();res=[]
    for n in range(1,max_len+1):
        for cols in combinations(features,n):
            g=data.groupby(list(cols),dropna=False).target_flag.agg(records='count',positives='sum').reset_index()
            g['positive_rate']=g.positives/g.records;g['lift']=g.positive_rate/max(base,1e-9)
            g=g[(g.records>=min_support)&(g.positives>=min_pos)&(g.positive_rate>=min_conf)&(g.lift>=min_lift)]
            for _,r in g.iterrows():
                lo,hi=wilson(int(r.positives),int(r.records))
                res.append({'rule_length':n,'rule':' AND '.join(f'{c} = {r[c]}' for c in cols),'features':' | '.join(cols),
                            'records':int(r.records),'positives':int(r.positives),'positive_rate':float(r.positive_rate),
                            'lift':float(r.lift),'ci_low':lo,'ci_high':hi})
    if not res:return pd.DataFrame()
    o=pd.DataFrame(res);o['rule_score']=o.lift*o.positive_rate*np.log1p(o.records)*np.sqrt(o.positives)
    return o.sort_values(['rule_score','lift','records'],ascending=False).reset_index(drop=True)

combination_rules=mine_combinations(rule_frame,MAX_RULE_LENGTH,MIN_SUPPORT_COUNT,MIN_POSITIVE_COUNT,MIN_CONFIDENCE,MIN_LIFT)
display(combination_rules.head(100))


## 9. FP-Growth association rules

In [ ]:

parts=[]
for c in [x for x in rule_frame.columns if x!='target_flag']:
    parts.append(pd.get_dummies(rule_frame[c].astype(str).map(lambda x:f'{c}={x}'),prefix='',prefix_sep='').astype(bool))
basket=pd.concat(parts,axis=1);basket['TARGET_POSITIVE']=rule_frame.target_flag.astype(bool).values
frequent_itemsets=fpgrowth(basket,min_support=ASSOCIATION_MIN_SUPPORT,use_colnames=True,max_len=ASSOCIATION_MAX_LEN)
assoc_all=association_rules(frequent_itemsets,metric='confidence',min_threshold=ASSOCIATION_MIN_CONFIDENCE)
association_rules_positive=assoc_all[assoc_all.consequents.apply(lambda x:len(x)==1 and 'TARGET_POSITIVE' in x)].copy()
association_rules_positive=association_rules_positive[association_rules_positive.lift>=ASSOCIATION_MIN_LIFT]
association_rules_positive['rule']=association_rules_positive.antecedents.apply(lambda x:' AND '.join(sorted(x)))+' => TARGET_POSITIVE'
association_rules_positive=association_rules_positive.sort_values(['lift','confidence','support'],ascending=False)
display(association_rules_positive[['rule','support','confidence','lift','leverage','conviction']].head(100))


## 10. Decision-tree model and rule extraction

In [ ]:

model_features=[c for c in df.columns if c not in set([TARGET_COLUMN,'target_flag']+ID_COLUMNS+FREE_TEXT_COLUMNS+EXCLUDE_COLUMNS+date_cols)]
X=df[model_features].copy();y=df.target_flag.copy()
num=[c for c in model_features if pd.api.types.is_numeric_dtype(X[c])];cat=[c for c in model_features if c not in num]
prep=ColumnTransformer([('num',Pipeline([('imp',SimpleImputer(strategy='median'))]),num),
                        ('cat',Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('enc',OneHotEncoder(handle_unknown='ignore',min_frequency=5))]),cat)])
model=DecisionTreeClassifier(max_depth=TREE_MAX_DEPTH,min_samples_leaf=TREE_MIN_SAMPLES_LEAF,class_weight='balanced',random_state=RANDOM_STATE)
pipe=Pipeline([('prep',prep),('model',model)])
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=TEST_SIZE,stratify=y,random_state=RANDOM_STATE)
pipe.fit(X_train,y_train);pred=pipe.predict(X_test);proba=pipe.predict_proba(X_test)[:,1]
print(classification_report(y_test,pred));print('ROC-AUC',roc_auc_score(y_test,proba),'PR-AUC',average_precision_score(y_test,proba));print(confusion_matrix(y_test,pred))
feature_names=pipe.named_steps['prep'].get_feature_names_out()
print(export_text(pipe.named_steps['model'],feature_names=list(feature_names),decimals=3))

def extract_rules(tree,names):
    t=tree.tree_;out=[]
    def rec(node,conds):
        if t.feature[node]!=_tree.TREE_UNDEFINED:
            name=names[t.feature[node]];thr=t.threshold[node]
            rec(t.children_left[node],conds+[f'{name} <= {thr:.4f}']);rec(t.children_right[node],conds+[f'{name} > {thr:.4f}'])
        else:
            counts=t.value[node][0];total=counts.sum();pos=counts[1] if len(counts)>1 else 0
            out.append({'rule':' AND '.join(conds),'weighted_records':float(total),'weighted_positives':float(pos),'predicted_positive_rate':float(pos/total if total else 0),'predicted_class':int(np.argmax(counts))})
    rec(0,[]);return pd.DataFrame(out).sort_values(['predicted_positive_rate','weighted_records'],ascending=False)
tree_rules=extract_rules(pipe.named_steps['model'],list(feature_names));display(tree_rules)


## 11. Holdout validation and risk bands

In [ ]:

def apply_rule(data,rule):
    m=pd.Series(True,index=data.index)
    for cond in rule.split(' AND '):
        if ' = ' not in cond:return pd.Series(False,index=data.index)
        f,v=cond.split(' = ',1)
        if f not in data.columns:return pd.Series(False,index=data.index)
        m &= data[f].astype(str).eq(v)
    return m

train_idx,test_idx=train_test_split(rule_frame.index,test_size=TEST_SIZE,stratify=rule_frame.target_flag,random_state=RANDOM_STATE)
train_rf,test_rf=rule_frame.loc[train_idx],rule_frame.loc[test_idx]
train_rules=mine_combinations(train_rf,MAX_RULE_LENGTH,max(5,int(MIN_SUPPORT_COUNT*(1-TEST_SIZE))),max(2,int(MIN_POSITIVE_COUNT*(1-TEST_SIZE))),MIN_CONFIDENCE,MIN_LIFT)
vals=[]
for _,r in train_rules.head(500).iterrows():
    m=apply_rule(test_rf,r.rule);n=int(m.sum());p=int(test_rf.loc[m,'target_flag'].sum()) if n else 0;rate=p/n if n else np.nan
    vals.append({'rule':r.rule,'train_records':r.records,'train_positive_rate':r.positive_rate,'train_lift':r.lift,
                 'test_records':n,'test_positives':p,'test_positive_rate':rate,'test_lift':rate/max(test_rf.target_flag.mean(),1e-9) if n else np.nan,
                 'rate_drop':r.positive_rate-rate if n else np.nan})
validated_rules=pd.DataFrame(vals).sort_values(['test_lift','test_records'],ascending=False) if vals else pd.DataFrame()
display(validated_rules.head(100))

scored=X_test.copy();scored['actual']=y_test.values;scored['score']=proba
scored['risk_band']=pd.qcut(scored.score,q=10,labels=[f'D{i}' for i in range(1,11)],duplicates='drop')
risk_band_summary=scored.groupby('risk_band',observed=False).agg(records=('actual','size'),positives=('actual','sum'),average_score=('score','mean'),min_score=('score','min'),max_score=('score','max')).reset_index()
risk_band_summary['positive_rate']=risk_band_summary.positives/risk_band_summary.records;risk_band_summary['lift']=risk_band_summary.positive_rate/max(y_test.mean(),1e-9)
display(risk_band_summary)


## 12. Final business rules and export

In [ ]:

final_rules=combination_rules.copy()
if not final_rules.empty and not validated_rules.empty:
    final_rules=final_rules.merge(validated_rules[['rule','test_records','test_positives','test_positive_rate','test_lift','rate_drop']],on='rule',how='left')
if not final_rules.empty:
    def action(r):
        if r.get('test_records',0)>=10 and r.get('test_lift',0)>=2:return 'High-priority investigation / specialist review'
        if r.lift>=2:return 'Enhanced document and provider validation'
        if r.lift>=1.5:return 'Additional automated checks'
        return 'Monitor'
    final_rules['recommended_action']=final_rules.apply(action,axis=1)
    final_rules['rule_id']=[f'R{i:04d}' for i in range(1,len(final_rules)+1)]
    first=['rule_id','rule','features','rule_length','records','positives','positive_rate','lift','ci_low','ci_high','rule_score','test_records','test_positives','test_positive_rate','test_lift','rate_drop','recommended_action']
    final_rules=final_rules[[c for c in first if c in final_rules.columns]]
display(final_rules.head(100))

excel_path=os.path.join(OUTPUT_FOLDER,OUTPUT_EXCEL)
with pd.ExcelWriter(excel_path,engine='openpyxl') as w:
    profile.to_excel(w,sheet_name='Data_Profile',index=False)
    numeric_summary.to_excel(w,sheet_name='Numeric_EDA',index=False)
    univariate_rules.to_excel(w,sheet_name='Univariate_Lift',index=False)
    combination_rules.to_excel(w,sheet_name='Combination_Rules',index=False)
    if not association_rules_positive.empty: association_rules_positive[['rule','support','confidence','lift','leverage','conviction']].to_excel(w,sheet_name='Association_Rules',index=False)
    tree_rules.to_excel(w,sheet_name='Tree_Rules',index=False)
    if not validated_rules.empty: validated_rules.to_excel(w,sheet_name='Validated_Rules',index=False)
    risk_band_summary.to_excel(w,sheet_name='Risk_Bands',index=False)
    if not final_rules.empty: final_rules.to_excel(w,sheet_name='Final_Business_Rules',index=False)
    bin_metadata.to_excel(w,sheet_name='Bin_Metadata',index=False)
profile.to_csv(os.path.join(OUTPUT_FOLDER,'data_profile.csv'),index=False)
combination_rules.to_csv(os.path.join(OUTPUT_FOLDER,'combination_rules.csv'),index=False)
if not final_rules.empty: final_rules.to_csv(os.path.join(OUTPUT_FOLDER,'final_business_rules.csv'),index=False)
print('Created:',excel_path)



## 13. Operational interpretation

Prioritize rules with adequate support, high lift, narrow confidence intervals, acceptable holdout performance, low train-to-test degradation, clear business meaning, and no post-outcome leakage.

Typical health-insurance patterns include short policy tenure plus high claim-to-premium ratio, non-network provider plus high room-rent share, repeated provider/procedure combinations, missing clinical documentation plus high-cost claims, overlapping admissions, diagnosis/procedure incompatibility, and unusual utilization close to policy expiry.
